In [1]:
#from langchain_text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.retrievers import BM25Retriever
from openai import AzureOpenAI
import dotenv
import getpass
import os
import requests
import pathlib   # navigates the file system and opens files
import pprint    # for inspecting Iliad response messages
import json

In [2]:
dotenv.load_dotenv()

ILIAD_API_KEY = os.getenv('ILIAD_API_KEY')
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
ILIAD_URL_BASE = os.getenv('ILIAD_URL_BASE')

if not os.getenv("ILIAD_API_KEY"):
    os.environ["ILIAD_API_KEY"] = getpass.getpass("Enter your ILIAD API key: ")

def get_access_token():
    response = requests.post(
        url="https://federation.abbvie.com/as/token.oauth2",
        data={
            "grant_type": "client_credentials",
            "client_id": os.environ["CLIENT_ID"],
            "client_secret": os.environ["CLIENT_SECRET"]
        }
    )
    response.raise_for_status()
    data = response.json()
    token = data["access_token"]
    return f"Bearer {token}"

In [3]:
url = ILIAD_URL_BASE + "/api/v1/chat/gpt-4o-mini"
headers = {"x-api-key": ILIAD_API_KEY}
message = {
    "role": "user",
    "content": "what's two plus two?"
}
payload = {"messages": [message]}
resp = requests.post(url, json=payload, headers=headers)
print(resp.json())

{'response_id': 'f9d1aa88-2e05-4cb9-9714-6343a514ad21', 'completion': {'role': 'assistant', 'content': 'Two plus two equals four.'}, 'parsed': None, 'cost': 6.93e-06}


In [4]:
# Get new token from 

USER_TOKEN = os.getenv("USER_TOKEN")

print(USER_TOKEN)

..._dCF5eQBSuUAO2ZQJ2utgMuw3LO4_1SQWiszc8Gw6sA8OOJ1AcD7YP95N5BiGhde07a4qVTUHbq4CNuaPIa-1MVE4


In [19]:
client = AzureOpenAI(
    azure_endpoint=ILIAD_URL_BASE,
    api_key=ILIAD_API_KEY,
    api_version="2023-07-01-preview",
)

response = client.chat.completions.create(
    model="gpt-5-global",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Does Azure OpenAI support customer managed keys?"},
        {"role": "assistant", "content": "Yes, customer managed keys are supported by Azure OpenAI."},
        {"role": "user", "content": "Do other Azure AI services support this too?"}
    ]
)

print(response.json())

{"id":"chatcmpl-CiRcgTPUcBheGQsk0rIex74m4hMVP","choices":[{"finish_reason":"stop","index":0,"logprobs":null,"message":{"content":"Yes. Beyond Azure OpenAI, several other Azure AI services support customer‑managed keys (CMK) with Azure Key Vault, though support varies by service, region, and sometimes pricing tier.\n\nCommon Azure AI offerings that support CMK:\n- Azure AI Search (Cognitive Search): CMK for index data and skillset artifacts.\n- Azure AI Language (Text Analytics), Azure AI Speech, and Azure AI Vision: CMK for data at rest on supported resource types/regions.\n- Azure AI Document Intelligence (Form Recognizer): CMK for stored data and learned artifacts.\n- Azure Machine Learning: CMK for the workspace and most managed data/metadata (and related resources like storage, registry, etc.).\n\nServices with limited or no CMK support (or service-dependent): \n- Some specialized services such as Translator, Content Safety, and Video Indexer have had partial or no CMK support; che

/tmp/ipykernel_1599363/4163657147.py:17: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(response.json())


In [5]:
# Create a Source
resp = requests.post(
    url=f"{ILIAD_URL_BASE}/api/v1/sources",
    headers={
        "x-api-key": ILIAD_API_KEY,
        "x-user-token": USER_TOKEN
    },
    json={
        "source": "SPAT",
        "description": "SPAT Documents",
        "custom_fields": json.dumps({
          "owning_facility": {"type": "text"},
          "location": {"type": "text"}
        })
    }
)
resp.raise_for_status()

HTTPError: 400 Client Error: Bad Request for url: https://api-epic.ir-gateway.abbvienet.com/iliad/api/v1/sources

McpError: argument of length 0